# Fast conversion of whole-slide images

Same settings and **pixel-identical outputs** as `run_conversion.ipynb`, using much less
memory and converting several slides at once. Use the **Python (histology-pipeline)** kernel.

Handles every format the original converter does: `.ndpi`/`.ndp` (Hamamatsu), `.svs`
(Leica), `.mrxs` (3DHistech/P1000), `.scn`, `.qptiff`, `.tif`/`.tiff` (e.g. Roche
Ventana), `.dcm` (Pramana), `.vsi` (Olympus), `.czi` (Zeiss) and `.isyntax`/`.i2syntax`.

- **Streamed** (never held in memory whole): `.dcm`, all OpenSlide formats and whole-slide
  TIFFs. Measured: 2â€“9 GB per slide, against 9â€“94 GB for the original code.
- **Read whole, then streamed to the writers:** `.vsi`, `.czi`, iSyntax and plain TIFFs use
  the original readers, so they still need the decoded slide in memory (the saving step
  no longer adds copies on top). Use fewer workers for these.
- Every page (full resolution, all 6 pyramid levels, thumbnail) matches the original
  converter pixel for pixel; `tests/test_streaming_conversion.py` checks this.
- Files are written as `*.part` and renamed when complete, so a stopped run can simply be
  restarted: finished images are skipped and unfinished ones are redone.

In [ ]:
import sys
from pathlib import Path

# Make the conversion scripts importable regardless of where Jupyter was started.
HERE = Path.cwd()
SCRIPT_DIR = HERE if (HERE / 'streaming_conversion.py').exists() else HERE / '01_conversion_downsampling'
sys.path.insert(0, str(SCRIPT_DIR))

from streaming_conversion import convert_folder
print('Python:', sys.executable)

## Settings

- `folder_names`, `pixel_resolutions`, `save_ome` correspond by position, as in the original notebook.
- `pixel_resolutions` are in micrometres/pixel; `0` keeps native resolution.
- `save_ome`: `0` = plain TIFF, `1` = OME-TIFF.
- `output_path = None` writes the subfolders inside the input folder.
- `load_native_resolution`: `1` decodes full resolution (default, as the original); `0` lets
  OpenSlide formats read a coarser stored level when no output needs full resolution.
- `workers`: slides converted at once. Streamed formats need roughly 2â€“9 GB per slide, so
  on a 128 GB machine 4â€“6 is a safe range; for `.vsi`/`.czi`/iSyntax use 1â€“2. Lower it if
  another large job is running.
- `threads_per_slide`: 4 was the best value measured; more threads per slide barely help.

In [ ]:
input_path = r'\\10.17.182.53\kiemen-lab-data\Marta Pereira\Multi scanner project\Scanned images\Scanner types\Hamamatsu_S210_40x'
output_path = None
folder_names = ['2x', '40x']
pixel_resolutions = [5, 0.25]
save_ome = [0, 1]
scanner_name = 'Hamamatsu_S210'  # Use 'P1000' for P1000; its validated MPP calibration is then applied
scanner_manifest = None           # Optional CSV with image,scanner columns for mixed batches
load_native_resolution = 1        # 1 = native decoding; 0 = allow the source pyramid (faster)
workers = 4                       # Slides converted at the same time
threads_per_slide = 4

## Check the input

In [ ]:
import psutil
from streaming_conversion import SLIDE_TYPES

source = Path(input_path)
out_root = Path(output_path) if output_path else (source if source.is_dir() else source.parent)
files = [f for f in source.iterdir() if f.is_file()] if source.is_dir() else [source]
slides = sorted(f for f in files if f.name.lower().endswith(SLIDE_TYPES))

def done(slide):
    return all((out_root / folder / (slide.stem + ('.ome.tif' if ome else '.tif'))).exists()
               for folder, ome in zip(folder_names, save_ome))

remaining = [s for s in slides if not done(s)]
by_type = {}
for s in slides:
    by_type[s.suffix.lower()] = by_type.get(s.suffix.lower(), 0) + 1
print(f'{len(slides)} slide(s) {by_type}; {len(slides) - len(remaining)} already converted, {len(remaining)} to do')
for slide in remaining[:10]:
    print(f'   {slide.name}  ({slide.stat().st_size / 1e9:.2f} GB)')
if len(remaining) > 10:
    print(f'   ... and {len(remaining) - 10} more')
whole = [s.name for s in remaining if s.suffix.lower() in ('.vsi', '.czi', '.isyntax', '.i2syntax')]
if whole:
    print(f'NOTE: {len(whole)} .vsi/.czi/iSyntax slide(s) are read whole into memory; consider workers = 1-2')
print(f'Free RAM now: {psutil.virtual_memory().available / 1e9:.0f} GB for {workers} worker(s)')

## Run the conversion

In [ ]:
import time

started = time.time()
convert_folder(str(source), str(out_root), folder_names, pixel_resolutions, save_ome,
               workers=workers, threads=threads_per_slide,
               scanner=scanner_name, scanner_manifest=scanner_manifest,
               load_native_resolution=load_native_resolution)
print(f'Finished in {(time.time() - started) / 60:.1f} min')

## Results

In [ ]:
import csv

for folder in folder_names:
    outputs = sorted((out_root / folder).glob('*.tif'))
    print(f'{folder}: {len(outputs)} file(s), {sum(f.stat().st_size for f in outputs) / 1e9:.1f} GB')

# Per-image times of this run (one timing CSV per worker plus one for the batch)
rows = []
for log in (out_root / 'timings').glob('conversion_*.csv'):
    if log.stat().st_mtime >= started:
        with open(log, newline='', encoding='utf-8') as stream:
            rows += [r for r in csv.DictReader(stream) if 'streaming' in r['detail']]
for r in sorted(rows, key=lambda r: r['started_utc']):
    print(f"   {r['phase']:12} {r['image']:28} {float(r['seconds']):8.1f} s  {r['status']}")